# Phase 6: 2D Spectrogram Convolutional Autoencoder
In this notebook, we transform our 1D vibration signals into 2D Time-Frequency Spectrograms using STFT. 
Then, we build a PyTorch 2D Convolutional Autoencoder to detect anomalies using Computer Vision techniques!

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import signal
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

## 1. Generating Spectrograms
We convert a 1024-point 1D signal into a 32x32 pixel Spectrogram using `scipy.signal.spectrogram`.

In [ ]:
# Load raw 1D data
X = np.load("../data/processed/X.npy")
y = np.load("../data/processed/y.npy")
meta = pd.read_csv("../data/processed/metadata.csv")

# Take one normal and one faulty signal
normal_signal = X[y == 0][0]
fault_signal = X[y == 1][0]

def plot_spectrogram(window, title):
    f, t, Sxx = signal.spectrogram(window, fs=12000, nperseg=64, noverlap=32)
    Sxx = Sxx[:-1, :] # drop last freq bin to make it 32 freq bins
    Sxx_log = 10 * np.log10(Sxx + 1e-10)
    
    plt.figure(figsize=(6, 5))
    plt.pcolormesh(t, f[:-1], Sxx_log, shading='gouraud', cmap='viridis')
    plt.title(title)
    plt.ylabel('Frequency [Hz]')
    plt.xlabel('Time [sec]')
    plt.colorbar(label='Intensity [dB]')
    plt.show()

plot_spectrogram(normal_signal, "Spectrogram of Normal Bearing")
plot_spectrogram(fault_signal, "Spectrogram of Faulty Bearing")


## 2. 2D CNN Autoencoder Architecture
We use PyTorch `Conv2D` and `ConvTranspose2D` layers to compress and reconstruct the 32x32 images.

In [ ]:
class Conv2DAutoencoder(nn.Module):
    def __init__(self):
        super(Conv2DAutoencoder, self).__init__()
        
        # Encoder: 32x32 -> 16x16 -> 8x8 -> 4x4
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(16, 32, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.ReLU()
        )
        
        # Decoder: 4x4 -> 8x8 -> 16x16 -> 32x32
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(64, 32, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(32, 16, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(16, 1, kernel_size=3, stride=2, padding=1, output_padding=1)
        )
        
    def forward(self, x):
        x = self.encoder(x)
        x = self.decoder(x)
        return x

model = Conv2DAutoencoder()
print(model)